In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score


In [4]:
# =============================================================================
# Step 1: Define paths and load your data
# =============================================================================

# --- IMPORTANT: EDIT THESE PATHS TO MATCH YOUR PROJECT DIRECTORY ---
DATADIR = 'C:/Users/OWNER/Machine Learning/FYP/'
CSV_PATH = 'C:/Users/OWNER/Machine Learning/FYP/yoruba character.csv'

try:
    df = pd.read_csv(CSV_PATH)
    print("CSV file loaded successfully.")
    print("First 5 rows of the dataframe:")
    print(df.head())
except Exception as e:
    print(f"Error loading CSV: {e}")
    print("Please check that the CSV_PATH is correct.")
    exit() # Exit the script if the data can't be loaded


CSV file loaded successfully.
First 5 rows of the dataframe:
              image label
0  img/img001_1.png     A
1  img/img001_2.png     A
2  img/img001_3.png     A
3  img/img001_4.png     A
4  img/img001_5.png     A


In [5]:
# =============================================================================
# Step 2: Prepare data generators for training, validation, and testing
# =============================================================================

# --- IMPORTANT: Adjust these as needed ---
# Split your dataframe into training, validation, and test sets.
# Using a fixed seed for reproducibility.
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Create ImageDataGenerator instances
train_data_generator = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2)
data_generator = ImageDataGenerator(rescale=1./255) # No augmentation for validation/test sets

# Flow from dataframe to create data generators
training_data_frame = train_data_generator.flow_from_dataframe(
    dataframe=train_df,
    directory=DATADIR,
    x_col='image',
    y_col='label',
    target_size=(64, 64),
    class_mode='categorical'
)

validation_data_frame = data_generator.flow_from_dataframe(
    dataframe=val_df,
    directory=DATADIR,
    x_col='image',
    y_col='label',
    target_size=(64, 64),
    class_mode='categorical'
)

test_data_frame = data_generator.flow_from_dataframe(
    dataframe=test_df,
    directory=DATADIR,
    x_col='image',
    y_col='label',
    target_size=(64, 64),
    class_mode='categorical',
    shuffle=False # IMPORTANT: Do not shuffle the test set for correct evaluation
)


Found 4000 validated image filenames belonging to 50 classes.
Found 500 validated image filenames belonging to 50 classes.
Found 500 validated image filenames belonging to 50 classes.


In [6]:
# =============================================================================
# Step 3: Build and Compile the Keras Model (similar to your image_4a0e4f.png)
# =============================================================================

# Define the number of classes based on your data
NUM_CLASSES = len(df['label'].unique())

model = Sequential([
    # Add your convolutional layers here if you have them. A typical CNN starts with Conv2D.
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Layers from your screenshot
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


C:\Users\OWNER\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 12544)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │       6,423,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 50)                  │          25,650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,468,082 (24.67 MB)

 Trainable params: 6,468,082 (24.67 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# =============================================================================
# Step 4: Train the model
# =============================================================================

EPOCHS = 50
print(f"\nTraining the model for {EPOCHS} epochs...")
history = model.fit(
    training_data_frame,
    epochs=EPOCHS,
    validation_data=validation_data_frame
)
print("Model training completed.")



Training the model for 50 epochs...


C:\Users\OWNER\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 65s 505ms/step - accuracy: 0.1061 - loss: 4.0156 - val_accuracy: 0.5060 - val_loss: 1.9110
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 48s 385ms/step - accuracy: 0.4535 - loss: 1.9747 - val_accuracy: 0.6100 - val_loss: 1.3417
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 46s 368ms/step - accuracy: 0.6013 - loss: 1.3685 - val_accuracy: 0.6900 - val_loss: 1.0379
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 377ms/step - accuracy: 0.6755 - loss: 1.0697 - val_accuracy: 0.7460 - val_loss: 0.8732
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 47s 376ms/step - accuracy: 0.7363 - loss: 0.8030 - val_accuracy: 0.7460 - val_loss: 0.8516
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 48s 381ms/step - accuracy: 0.7654 - loss: 0.7083 - val_accuracy: 0.7740 - val_loss: 0.7964
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 48s 381ms/step - accuracy: 0.8104 - loss: 0.5759 - val_accuracy: 0.7860 - val_loss: 0.7430
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 46s 370ms/step - accuracy: 0.8352 - loss: 0

In [8]:
# =============================================================================
# Step 5: Evaluate the trained model on the test data
# =============================================================================

print("\n--- Evaluating Model on the Test Set ---")
# Make predictions
predictions = model.predict(test_data_frame, verbose=1)

# Get true labels and convert predictions to single class labels
true_labels = test_data_frame.classes
predicted_labels = np.argmax(predictions, axis=1)
class_names = list(test_data_frame.class_indices.keys())

# Generate a classification report for a comprehensive summary
print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=class_names))

# Calculate and print individual metrics
accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=0)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=0)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=0)

print(f"\nFinal Test Set Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")



--- Evaluating Model on the Test Set ---
16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 628ms/step

Classification Report:
              precision    recall  f1-score   support

           A       0.00      0.00      0.00        10
           B       0.00      0.00      0.00        10
           D       0.00      0.00      0.00        10
           E       0.00      0.00      0.00        10
           F       0.00      0.00      0.00        10
           G       0.00      0.00      0.00        10
          GB       0.00      0.00      0.00        10
           H       0.00      0.00      0.00        10
           I       0.00      0.00      0.00        10
           J       0.00      0.00      0.00        10
           K       0.00      0.00      0.00        10
           L       0.00      0.00      0.00        10
           M       0.00      0.00      0.00        10
           N       0.00      0.00      0.00        10
           O       0.00      0.00      0.00        10
           P       0.00    

C:\Users\OWNER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\OWNER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\OWNER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 12544)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │       6,423,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 50)                  │          25,650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,468,082 (24.67 MB)

 Trainable params: 6,468,082 (24.67 MB)

 Non-trainable params: 0 (0.00 B)